# Data Preparation

In [ ]:
"""
prepare_data.py — Pre-process raw GitHub Python files for autocomplete training.

Features:
  • Deduplication by MD5 hash
  • Quality filtering (min lines, max line length, syntax check)
  • Train / val / test split with stratification by file length
  • Statistics report + histogram saved to PNG
"""

import os, re, sys, glob, json, hashlib, ast, random, argparse, shutil
from pathlib import Path
from typing import List, Tuple, Dict
from collections import Counter
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


def md5(text: str) -> str:
    return hashlib.md5(text.encode("utf-8", errors="replace")).hexdigest()


def is_valid_python(text: str) -> bool:
    try:
        ast.parse(text)
        return True
    except SyntaxError:
        return False


def basic_quality(text: str,
                  min_lines: int = 10,
                  max_lines: int = 10_000,
                  max_avg_line_len: int = 200,
                  min_code_ratio: float = 0.75) -> bool:
    lines = text.splitlines()
    if len(lines) < min_lines or len(lines) > max_lines:
        return False
    avg_len = sum(len(l) for l in lines) / max(1, len(lines))
    if avg_len > max_avg_line_len:
        return False
    # ratio of non-blank, non-comment lines
    code_lines = [l for l in lines
                  if l.strip() and not l.strip().startswith("#")]
    if len(code_lines) / max(1, len(lines)) < min_code_ratio:
        return False
    return True


def clean(text: str) -> str:
    """Light normalisation: strip trailing spaces, normalise line endings."""
    lines = text.splitlines()
    lines = [l.rstrip() for l in lines]
    # remove excessively long lines (binary / generated)
    lines = [l for l in lines if len(l) <= 500]
    return "\n".join(lines)


# ─────────────────────────────────────────────────────────────
# Stats + visualisation
# ─────────────────────────────────────────────────────────────

def compute_stats(texts: List[str]) -> Dict:
    line_counts = [len(t.splitlines()) for t in texts]
    char_counts = [len(t) for t in texts]
    token_approx = [len(t.split()) for t in texts]
    return dict(n=len(texts),
                line_counts=line_counts,
                char_counts=char_counts,
                token_approx=token_approx)


def plot_stats(stats: Dict, out_path: str) -> None:
    DARK = "#0d1117"
    MID = "#161b22"
    GRID = "#21262d"
    BLUE = "#58a6ff"
    GREEN = "#3fb950"
    ORG = "#ffa657"
    TXT = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor":  MID, "axes.edgecolor": GRID,
        "axes.labelcolor": TXT, "xtick.color": TXT,
        "ytick.color": TXT, "text.color": TXT, "grid.color": GRID,
    })

    fig = plt.figure(figsize=(16, 8), facecolor=DARK)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    def hist(ax, data, color, label, xlabel):
        # data = {item: data.count(item) for item in set(data)}
        # x = list(data.keys())
        # height = [data[elem] for elem in x]
        # ax.bar(x, height, color=color, alpha=0.85, edgecolor=GRID)
        ax.hist(data, color=color, alpha=0.85, edgecolor=GRID)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        # ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(label, fontsize=10, color=BLUE, pad=6)
        ax.grid(True, lw=0.5)
        ax.axvline(np.median(data), color="white", lw=1.2, linestyle="--",
                   label=f"median={np.median(data):.0f}")
        ax.legend(fontsize=8)

    hist(fig.add_subplot(gs[0, 0]), stats["line_counts"],  BLUE,  "Lines per file",   "Lines")
    hist(fig.add_subplot(gs[0, 1]), stats["char_counts"],  GREEN, "Chars per file",   "Characters")
    hist(fig.add_subplot(gs[0, 2]), stats["token_approx"], ORG,   "Tokens per file",  "Tokens (approx)")

    # cumulative
    for ax_pos, data, color, label in [
        (gs[1, 0], stats["line_counts"],  BLUE,  "CDF — Lines"),
        (gs[1, 1], stats["char_counts"],  GREEN, "CDF — Chars"),
        (gs[1, 2], stats["token_approx"], ORG,   "CDF — Tokens"),
    ]:
        ax = fig.add_subplot(ax_pos)
        s = sorted(data)
        ax.plot(s, np.linspace(0, 1, len(s)), color=color, lw=1.8)
        ax.set_xlabel("Value", fontsize=9)
        # ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_ylabel("Cumulative fraction", fontsize=9)
        ax.set_title(label, fontsize=10, color=BLUE, pad=6)
        ax.grid(True, lw=0.5)

    fig.suptitle(f"Dataset Statistics — {stats['n']:,} files",
                 fontsize=13, color=BLUE, y=1.02)
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Stats] plot saved → {out_path}")

def prepare(arguments: Arguments) -> None:
    print(f"[Prepare] scanning {arguments.raw_dir} …")
    paths = glob.glob(os.path.join(arguments.raw_dir, "**/*.py"), recursive=True)
    print(f"[Prepare] found {len(paths):,} .py files")
    if arguments.delete_previous:
        if not Path.exists(Path(fr"{arguments.out_dir}")):
            print(f"[Deleting] Output directory does not exist")
        else:
            to_del_paths = glob.glob(os.path.join(arguments.out_dir, "**/*.py"), recursive=True)
            print(f"[Deleting] found {len(to_del_paths):,} .py files")
            shutil.rmtree(arguments.out_dir)
            print(f"[Deleting] Previous cleaned dataset was deleted")
            
        
    
    seen_hashes: set = set()
    accepted: List[str] = []
    stats = Counter({"total": 0, "dup": 0, "quality": 0, "ok": 0})

    for fp in paths:
        stats["total"] += 1
        try:
            raw = Path(fp).read_text(errors="replace")
        except Exception:
            continue

        # dedup
        h = md5(raw)
        if h in seen_hashes:
            stats["dup"] += 1
            continue
        seen_hashes.add(h)

        # quality
        if not basic_quality(raw, arguments.min_lines, arguments.max_lines):
            stats["quality"] += 1
            continue

        accepted.append(clean(raw))
        stats["ok"] += 1

        if arguments.max_files and len(accepted) >= arguments.max_files:
            break

    print(f"[Prepare] filter summary: {dict(stats)}")
    print(f"[Prepare] kept {len(accepted):,} files")

    # stats + plot
    s = compute_stats(accepted)
    os.makedirs(arguments.out_dir, exist_ok=True)
    plot_stats(s, os.path.join(arguments.out_dir, "dataset_stats.png"))

    # split
    random.seed(arguments.seed)
    random.shuffle(accepted)
    n = len(accepted)
    n_val = max(1, int(n * arguments.val_frac))
    n_test = max(1, int(n * arguments.test_frac))
    n_train = n - n_val - n_test

    splits = {
        "train": accepted[:n_train],
        "val": accepted[n_train: n_train + n_val],
        "test": accepted[n_train + n_val:],
    }
    for name, texts in splits.items():
        split_dir = os.path.join(arguments.out_dir, name)
        os.makedirs(split_dir, exist_ok=True)
        for i, text in enumerate(texts):
            with open(os.path.join(split_dir, f"file_{i:06d}.py"), 'w', encoding='utf-8') as file:
                file.write(text)
        print(f"[Prepare] {name}: {len(texts):,} files → {split_dir}")

    # write summary json
    summary = {
        "total_raw": stats["total"],
        "kept": stats["ok"],
        "train": n_train,
        "val": n_val,
        "test": n_test,
        "median_lines": int(np.median(s["line_counts"])),
        "median_chars": int(np.median(s["char_counts"])),
    }
    with open(os.path.join(arguments.out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"[Prepare] summary → {arguments.out_dir}/summary.json")


class Arguments():
    def __init__(self, raw_dir: str = 'Raw_Dataset', out_dir: str = 'Clean_Dataset', 
                    min_lines: int = 10, max_lines: int = 10_000, max_files: int = 0,
                    val_frac: float = 0.05, test_frac: float = 0.05, seed: int = 42,
                    delete_previous: bool = True) -> None:
        self.raw_dir = raw_dir
        self.out_dir = out_dir
        self.min_lines = min_lines
        self.max_lines = max_lines
        self.max_files = max_files
        self.val_frac = val_frac
        self.test_frac = test_frac
        self.seed = seed
        self.delete_previous = delete_previous



prepare(Arguments(min_lines=25, max_lines=400))
# prepare(Arguments(raw_dir="Test_Dataset", out_dir="Test_Dataset_Output"))

[Prepare] scanning Raw_Dataset …
[Prepare] found 35,197 .py files
[Deleting] found 12,110 .py files
[Deleting] Previous cleaned dataset was deleted
[Prepare] filter summary: {'total': 35197, 'dup': 3482, 'quality': 19605, 'ok': 12110}
[Prepare] kept 12,110 files
[Stats] plot saved → Clean_Dataset\dataset_stats.png
[Prepare] train: 10,900 files → Clean_Dataset\train
[Prepare] val: 605 files → Clean_Dataset\val
[Prepare] test: 605 files → Clean_Dataset\test
[Prepare] summary → Clean_Dataset/summary.json